# Imputation Environment Check

Run the next cell to verify that required libraries are available for:
- Mean imputation (`SimpleImputer`)
- kNN imputation (`KNNImputer`)
- MICE and SoftImpute workflows (via `hyperimpute`)
- Research repos (`GRAPE`, `DiffPuter`)

In [1]:
import sys
import pathlib
import importlib
from importlib import metadata

# ---- self-contained DiffPuter path setup ----------------------------------
# This makes the cell work on its own. If you also have a separate path-setup
# cell, that's fine — running it twice is a no-op.
HERE = pathlib.Path.cwd()
for cand in [
    HERE / "DiffPuter",
    HERE / "external" / "DiffPuter",
    HERE.parent / "DiffPuter",
]:
    if (cand / "main.py").is_file():
        for p in [cand, cand / "baselines", cand / "baselines" / "GRAPE"]:
            sp = str(p.resolve())
            if sp not in sys.path:
                sys.path.insert(0, sp)
        break

# Module name (for import) -> distribution name (for version lookup).
# These differ for some packages (sklearn vs scikit-learn, yaml vs PyYAML,
# ot vs POT).
required_modules = {
    # core scientific
    "numpy":         "numpy",
    "pandas":        "pandas",
    "scipy":         "scipy",
    "sklearn":       "scikit-learn",
    "matplotlib":    "matplotlib",
    "statsmodels":   "statsmodels",

    # baselines + SOTA
    "fancyimpute":   "fancyimpute",     # Mean (SimpleFill), kNN, SoftImpute
    "hyperimpute":   "hyperimpute",     # HyperImpute + MICE plugin

    # deep learning + GRAPE deps
    "torch":             "torch",
    "torch_geometric":   "torch_geometric",
    "h5py":              "h5py",
    "networkx":          "networkx",

    # DiffPuter deps
    "ot":            "POT",             # POT distributes as POT, imports as ot
    "FrEIA":         "FrEIA",
    "timm":          "timm",
    "yaml":          "PyYAML",

    # notebook
    "ipykernel":     "ipykernel",
}

# Modules that are nice-to-have but the env still works without them.
optional_modules = {
    "torch_scatter": "torch_scatter",   # GRAPE only
}

# Note: GRAPE and DiffPuter are loaded as local source clones, not pip packages.


def check_module(module_name: str, package_name: str):
    try:
        importlib.import_module(module_name)
    except Exception as e:
        return "MISSING", str(e)
    try:
        version = metadata.version(package_name)
    except metadata.PackageNotFoundError:
        version = "unknown"
    return "OK", version


print("Required packages")
print("-" * 60)
missing = []
for module_name, package_name in required_modules.items():
    status, info = check_module(module_name, package_name)
    marker = "✓" if status == "OK" else "✗"
    print(f"  {marker} {package_name:18} {status:8} {info}")
    if status == "MISSING":
        missing.append(package_name)

print("\nOptional packages")
print("-" * 60)
for module_name, package_name in optional_modules.items():
    status, info = check_module(module_name, package_name)
    marker = "✓" if status == "OK" else "○"
    print(f"  {marker} {package_name:18} {status:8} {info}")

print("\nSmoke-test imports for imputation APIs")
print("-" * 60)

api_checks = [
    ("Mean / kNN (sklearn)",
     lambda: __import__("sklearn.impute", fromlist=["SimpleImputer", "KNNImputer"])),
    ("MICE (sklearn IterativeImputer)",
     lambda: (
         __import__("sklearn.experimental", fromlist=["enable_iterative_imputer"]),
         __import__("sklearn.impute",       fromlist=["IterativeImputer"]),
     )),
    ("SoftImpute / KNN / SimpleFill (fancyimpute)",
     lambda: __import__("fancyimpute", fromlist=["SoftImpute", "KNN", "SimpleFill"])),
    ("HyperImpute (hyperimpute.plugins.imputers.Imputers)",
     lambda: __import__("hyperimpute.plugins.imputers", fromlist=["Imputers"])),
    ("PyTorch Geometric (for GRAPE)",
     lambda: __import__("torch_geometric")),
]

for label, fn in api_checks:
    try:
        fn()
        print(f"  ✓ {label}: OK")
    except Exception as e:
        print(f"  ✗ {label}: FAILED | {e}")

print("\nLocal repo modules (DiffPuter / GRAPE)")
print("-" * 60)
repo_checks = [
    ("DiffPuter dataset",   "dataset",          ["load_dataset", "get_eval", "mean_std"]),
    ("DiffPuter diffusion", "diffusion_utils",  ["sample_step", "impute_mask", "EDMLoss"]),
    ("DiffPuter model",     "model",            ["MLPDiffusion", "Model"]),
    ("GRAPE training",      "training.gnn_mdi", ["train_gnn_mdi"]),
]
for label, mod, names in repo_checks:
    try:
        m = importlib.import_module(mod)
        for n in names:
            getattr(m, n)
        print(f"  ✓ {label}: OK ({mod})")
    except Exception as e:
        print(f"  ✗ {label}: FAILED | {type(e).__name__}: {e}")

print()
if missing:
    print(f"⚠  Missing required packages: {', '.join(missing)}")
    print("   Re-run: pip install -r requirements-base.txt")
else:
    print("All required packages present.")

Required packages
------------------------------------------------------------
  ✓ numpy              OK       1.26.4
  ✓ pandas             OK       2.3.3
  ✓ scipy              OK       1.17.1
  ✓ scikit-learn       OK       1.6.1
  ✓ matplotlib         OK       3.10.9
  ✓ statsmodels        OK       0.14.6
  ✓ fancyimpute        OK       0.7.0
  ✓ hyperimpute        OK       0.1.17
  ✓ torch              OK       2.4.1
  ✓ torch_geometric    OK       2.7.0
  ✓ h5py               OK       3.14.0
  ✓ networkx           OK       3.6.1


I0000 00:00:1778346205.899405   53472 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


  ✓ POT                OK       0.9.6.post1
  ✓ FrEIA              OK       0.2
  ✓ timm               OK       1.0.27
  ✓ PyYAML             OK       6.0.3
  ✓ ipykernel          OK       7.2.0

Optional packages
------------------------------------------------------------
  ✓ torch_scatter      OK       2.1.2+pt24cu121

Smoke-test imports for imputation APIs
------------------------------------------------------------
  ✓ Mean / kNN (sklearn): OK
  ✓ MICE (sklearn IterativeImputer): OK
  ✓ SoftImpute / KNN / SimpleFill (fancyimpute): OK
  ✓ HyperImpute (hyperimpute.plugins.imputers.Imputers): OK
  ✓ PyTorch Geometric (for GRAPE): OK

Local repo modules (DiffPuter / GRAPE)
------------------------------------------------------------
  ✓ DiffPuter dataset: OK (dataset)
  ✓ DiffPuter diffusion: OK (diffusion_utils)
  ✓ DiffPuter model: OK (model)
  ✓ GRAPE training: OK (training.gnn_mdi)

All required packages present.


# Dataset Analysis
Perform simple data exploration: Shape, Columns, Data types, Missing values, First few rows

In [2]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

# Load the dataset
df = pd.read_csv('Scenario5/scenario5.csv')

# Drop unnamed columns
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]

# Convert relative paths to absolute for reference
base_dir = Path('Scenario5')

# Data exploration
print(f"Shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nData types:\n{df.dtypes}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nFirst few rows:")
df.head()

Shape: (2300, 15)

Columns: ['index', 'unit1_rgb', 'unit1_pwr_60ghz', 'unit1_loc', 'unit2_loc', 'unit1_beam_index', 'seq_index', 'time_stamp[UTC]', 'unit2_direction', 'unit2_num_sat', 'unit2_sat_used', 'unit2_fix_type', 'unit2_DGPS', 'unit2_PDOP', 'unit2_HDOP']

Data types:
index                 int64
unit1_rgb            object
unit1_pwr_60ghz      object
unit1_loc            object
unit2_loc            object
unit1_beam_index      int64
seq_index             int64
time_stamp[UTC]      object
unit2_direction       int64
unit2_num_sat         int64
unit2_sat_used       object
unit2_fix_type       object
unit2_DGPS           object
unit2_PDOP          float64
unit2_HDOP          float64
dtype: object

Missing values:
index               0
unit1_rgb           0
unit1_pwr_60ghz     0
unit1_loc           0
unit2_loc           0
unit1_beam_index    0
seq_index           0
time_stamp[UTC]     0
unit2_direction     0
unit2_num_sat       0
unit2_sat_used      0
unit2_fix_type      0
unit2_DGPS

,index,unit1_rgb,unit1_pwr_60ghz,unit1_loc,unit2_loc,unit1_beam_index,seq_index,time_stamp[UTC],unit2_direction,unit2_num_sat,unit2_sat_used,unit2_fix_type,unit2_DGPS,unit2_PDOP,unit2_HDOP
0,1,./unit1/camera_data/image_BS1_259_03_20_31.jpg,./unit1/mmWave_data/mmWave_power_0.txt,./unit1/GPS_data/gps_location.txt,./unit2/GPS_data/gps_location_0.txt,60,1,['03-20-31-142'],1,21,G2 G5 G12 G18 G25 G29 R5 R6 R7 R10 R12 R20 R21...,3D,Yes,1.1,0.6
1,2,./unit1/camera_data/image_BS1_260_03_20_31.jpg,./unit1/mmWave_data/mmWave_power_1.txt,./unit1/GPS_data/gps_location.txt,./unit2/GPS_data/gps_location_1.txt,60,1,['03-20-31-284'],1,24,G2 G5 G12 G18 G25 G29 R5 R6 R7 R10 R12 R20 R21...,3D,Yes,1.1,0.6
2,3,./unit1/camera_data/image_BS1_261_03_20_31.jpg,./unit1/mmWave_data/mmWave_power_2.txt,./unit1/GPS_data/gps_location.txt,./unit2/GPS_data/gps_location_2.txt,58,1,['03-20-31-426'],1,14,G2 G5 G12 G18 G25 G29 R5 R6 R7 R10 R12 R20 R21...,3D,Yes,1.1,0.6
3,4,./unit1/camera_data/image_BS1_262_03_20_31.jpg,./unit1/mmWave_data/mmWave_power_3.txt,./unit1/GPS_data/gps_location.txt,./unit2/GPS_data/gps_location_3.txt,59,1,['03-20-31-568'],1,21,G2 G5 G12 G18 G25 G29 R5 R6 R7 R10 R12 R20 R21...,3D,Yes,1.1,0.6
4,5,./unit1/camera_data/image_BS1_263_03_20_31.jpg,./unit1/mmWave_data/mmWave_power_4.txt,./unit1/GPS_data/gps_location.txt,./unit2/GPS_data/gps_location_4.txt,60,1,['03-20-31-710'],1,21,G2 G5 G12 G18 G25 G29 R5 R6 R7 R10 R12 R20 R21...,3D,Yes,1.1,0.6


# Dataset preprocessing
- Convert path columns into value columns
- Drop redundant/non-usable columns

In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

np.random.seed(42)

# Parse location .txt files into lat/lon columns
base_dir = Path('Scenario5')

def parse_loc_file(rel_path):
    """
    Read a location .txt file and return (lat, lon) as floats. Handles both comma- and whitespace-separated formats.
    """
    if pd.isna(rel_path):
        return np.nan, np.nan
    try:
        full_path = base_dir / rel_path if not Path(rel_path).is_absolute() else Path(rel_path)
        with open(full_path, 'r') as f:
            content = f.read().strip()
        # Handle comma- or whitespace-separated
        parts = [p.strip() for p in content.replace(',', ' ').split()]
        lat, lon = float(parts[0]), float(parts[1])
        return lat, lon
    except (FileNotFoundError, ValueError, IndexError) as e:
        return np.nan, np.nan

# Peek at one file first to confirm format before parsing all 2300
sample_path = base_dir / df['unit1_loc'].iloc[0]
print(f"Sample loc file ({sample_path}):")
with open(sample_path, 'r') as f:
    print(repr(f.read()))

df[['unit2_lat', 'unit2_lon']] = df['unit2_loc'].apply(
    lambda p: pd.Series(parse_loc_file(p))
)

print(f"\nParsed coordinates — first few rows:")
print(df[['unit2_lat', 'unit2_lon']].head())
print(f"\nParse failures (NaNs) per column:")
print(df[['unit2_lat', 'unit2_lon']].isna().sum())

Sample loc file (Scenario5/unit1/GPS_data/gps_location.txt):
'33.42069305555555\n-111.929175\n\n\n'

Parsed coordinates — first few rows:
   unit2_lat   unit2_lon
0  33.420483 -111.928924
1  33.420488 -111.928924
2  33.420492 -111.928924
3  33.420497 -111.928925
4  33.420502 -111.928925

Parse failures (NaNs) per column:
unit2_lat    0
unit2_lon    0
dtype: int64


In [4]:
import pandas as pd
import numpy as np
from pathlib import Path

base_dir = Path('Scenario5')
N_BEAMS = 64

def parse_pwr_file(rel_path):
    """
    Read an mmWave power .txt file and return a length-64 numpy array.
    Each line is one beam's received power (scientific notation).
    Returns array of NaNs on failure.
    """
    if pd.isna(rel_path):
        return np.full(N_BEAMS, np.nan)
    try:
        full_path = base_dir / rel_path if not Path(rel_path).is_absolute() else Path(rel_path)
        vals = np.loadtxt(full_path)
        if vals.shape != (N_BEAMS,):
            # Unexpected shape — flag it rather than silently truncating
            return np.full(N_BEAMS, np.nan)
        return vals
    except (FileNotFoundError, ValueError):
        return np.full(N_BEAMS, np.nan)

# Parse all files into a (n_rows, 64) array — fast vectorised assignment
print("Parsing mmWave power files...")
pwr_matrix = np.vstack([parse_pwr_file(p) for p in df['unit1_pwr_60ghz']])
print(f"Power matrix shape: {pwr_matrix.shape}")
print(f"Files that failed to parse: {np.isnan(pwr_matrix).all(axis=1).sum()}")

# Expand into 64 columns: beam_00, beam_01, ..., beam_63
beam_cols = [f'beam_{i:02d}' for i in range(N_BEAMS)]
pwr_df = pd.DataFrame(pwr_matrix, columns=beam_cols, index=df.index)

# Check if the beam with max power matches the recorded optimal beam index (unit1_beam_index)
parsed_argmax = np.argmax(pwr_matrix, axis=1) + 1
match_rate = (parsed_argmax == df['unit1_beam_index'].values).mean()
print(f"Argmax matches unit1_beam_index: {match_rate*100:.2f}%")

# Build clean tabular feature set: 64 beams + GPS quality + receiver location
df_add_pwr = pd.concat([
    pwr_df,                                                                  # 64 beam columns
    df
], axis=1)

print(f"\nFinal feature shape: {df_add_pwr.shape}")
print(f"\nMissing values before injection: {df_add_pwr.isna().sum().sum()}")

Parsing mmWave power files...
Power matrix shape: (2300, 64)
Files that failed to parse: 0
Argmax matches unit1_beam_index: 100.00%

Final feature shape: (2300, 81)

Missing values before injection: 0


### Dropped columns

`index` — just a row counter (1, 2, 3...). Perfectly correlated with row position, carries no information.

`unit1_rgb` — image file paths. Not tabular data. Visual features would require a separate embedding pipeline and are out of scope for tabular imputation.

`unit1_pwr_60ghz` — original file paths to mmWave power readings. Replaced by 64 parsed `beam_XX` columns containing the actual per-beam received power values.

`unit1_loc` — paths to GPS files for the static basestation. Unit1 is stationary, so after parsing the coordinates are constant across all rows. A constant column has zero variance and zero predictive value; including it would also give imputers a free 100% accuracy on that column, artificially inflating overall metrics.

`unit2_loc` — original file paths for the mobile receiver's GPS readings. Replaced by parsed `unit2_lat` and `unit2_lon` columns.

`unit1_beam_index` — the optimal beam index, computed as `argmax(beam_00..beam_63) + 1`. Including it alongside the 64 beam columns would be perfect leakage: any model could trivially recover it from the beam values. Held aside as the evaluation target (`y_optimal_beam`) for downstream "did imputation preserve the optimal beam?" analysis.

`seq_index` — sequence identifier; rows belonging to the same trajectory share a value. Metadata for grouping rather than a feature. Dropped for imputation experiments but will be retained separately for sequence-aware train/test splits in downstream modelling.

`time_stamp[UTC]` — within-second timestamp in `'03-20-31-142'` format. Not directly usable as a tabular feature without further engineering (e.g. time-of-day, inter-row deltas), and not needed for the missingness experiments.

`unit2_sat_used` — string-encoded list of locked satellite IDs (e.g. "G2 G5 G12 G18 G25 G29 R5 R6..."). Tabular use would require one-hot encoding every possible satellite (dozens of columns) or a count. We already have `unit2_num_sat` as the count, making this column redundant.

`unit2_fix_type` — constant value (`"3D"`) across all rows. Zero variance, no information.

`unit2_DGPS` — constant value (`"Yes"`) across all rows. Zero variance, no information.

### Retained columns

64 `beam_00`..`beam_63` columns — per-beam received power at 60 GHz. The core signal for 6G beam-prediction work. Adjacent beams are spatially correlated (smooth angular response with a main lobe), which gives correlation-aware imputers a meaningful edge over naive ones.

`unit2_lat`, `unit2_lon` — parsed coordinates of the mobile receiver. Vary across rows and correlate with which beam is geometrically optimal.

`unit2_direction` — direction of motion (0 or 1). Binary but non-constant; informative spatial context.

`unit2_num_sat`, `unit2_PDOP`, `unit2_HDOP` — GPS quality indicators reflecting satellite visibility and geometric dilution of precision. Useful as side information for imputation, since poor GPS conditions tend to correlate with poor mmWave link quality (multipath, obstructions).

In [5]:
print (df_add_pwr['unit2_DGPS'].nunique()) # Drop unit2_DGPS since it has only one unique value and thus provides no useful information for imputation.
print(df_add_pwr['unit2_fix_type'].nunique()) # Drop unit2_fix_type since it has only one unique value and thus provides no useful information for imputation.

print(df.columns)

drop_columns = ['index', 'unit1_rgb', 'unit1_pwr_60ghz', 'unit1_loc', 'unit2_loc', 'unit1_beam_index','seq_index', 'time_stamp[UTC]', 'unit2_sat_used', 'unit2_fix_type', 'unit2_DGPS']

clean_data = df_add_pwr.drop(columns=drop_columns)

print("Remaining columns in clean_data:")
print(clean_data.columns)
print(clean_data)

# Total missing values after parsing but before injection
total_missing = clean_data.isna().sum().sum()
print(f"\nTotal missing values after parsing: {total_missing}")

1
1
Index(['index', 'unit1_rgb', 'unit1_pwr_60ghz', 'unit1_loc', 'unit2_loc',
       'unit1_beam_index', 'seq_index', 'time_stamp[UTC]', 'unit2_direction',
       'unit2_num_sat', 'unit2_sat_used', 'unit2_fix_type', 'unit2_DGPS',
       'unit2_PDOP', 'unit2_HDOP', 'unit2_lat', 'unit2_lon'],
      dtype='object')
Remaining columns in clean_data:
Index(['beam_00', 'beam_01', 'beam_02', 'beam_03', 'beam_04', 'beam_05',
       'beam_06', 'beam_07', 'beam_08', 'beam_09', 'beam_10', 'beam_11',
       'beam_12', 'beam_13', 'beam_14', 'beam_15', 'beam_16', 'beam_17',
       'beam_18', 'beam_19', 'beam_20', 'beam_21', 'beam_22', 'beam_23',
       'beam_24', 'beam_25', 'beam_26', 'beam_27', 'beam_28', 'beam_29',
       'beam_30', 'beam_31', 'beam_32', 'beam_33', 'beam_34', 'beam_35',
       'beam_36', 'beam_37', 'beam_38', 'beam_39', 'beam_40', 'beam_41',
       'beam_42', 'beam_43', 'beam_44', 'beam_45', 'beam_46', 'beam_47',
       'beam_48', 'beam_49', 'beam_50', 'beam_51', 'beam_52', 'beam_5

# Perform  MCAR (10%, 30% , 50%)

In [6]:
# Libraries for imputation
from sklearn.impute import SimpleImputer, KNNImputer, IterativeImputer
from sklearn.preprocessing import StandardScaler
from fancyimpute import SoftImpute, KNN, SimpleFill
from hyperimpute.plugins.imputers import Imputers

In [7]:
PROPORTIONS = [0.10, 0.30, 0.50]
N_SEEDS = 100
 
BEAM_COLS = [f"beam_{i:02d}" for i in range(64)]
GPS_COLS = [
    "unit2_lat",
    "unit2_lon",
    "unit2_direction",
    "unit2_num_sat",
    "unit2_PDOP",
    "unit2_HDOP",
]
RETAINED_COLS = BEAM_COLS + GPS_COLS

def validate_dataframe(df: pd.DataFrame) -> None:
    """Raise a clear error if expected columns are missing."""
    missing = [c for c in RETAINED_COLS if c not in df.columns]
    if missing:
        raise KeyError(
            f"clean_data is missing {len(missing)} expected column(s): "
            f"{missing[:5]}{'...' if len(missing) > 5 else ''}"
        )
    if df[RETAINED_COLS].isna().any().any():
        raise ValueError(
            "clean_data already contains NaN values in retained columns. "
            "Amputation experiments require fully observed input."
        )
        

def ampute_mcar(data_array: np.ndarray, prop: float, target_col_idx: np.ndarray,
                rng: np.random.Generator) -> np.ndarray:
    """
    Generate an MCAR boolean mask of shape `data_array.shape`.
    Only positions in `target_col_idx` are eligible to be amputed; other
    columns always remain observed.
 
    Each eligible cell is independently dropped with probability `prop`.
    """
    mask = np.zeros(data_array.shape, dtype=bool)
    sub_mask = rng.random((data_array.shape[0], len(target_col_idx))) < prop
    mask[:, target_col_idx] = sub_mask
    return mask
 
 
def evaluate_one_run(data_scaled: np.ndarray, mask: np.ndarray,
                     beam_idx: np.ndarray, gps_idx: np.ndarray) -> dict:
    """
    Apply mask -> mean impute -> compute RMSE on amputed cells.
 
    Returns dict with global RMSE and per-group RMSE (beams, gps).
    Per-group RMSE is NaN if no cells in that group were amputed.
    """
    # Apply mask: copy data and set masked cells to NaN
    data_masked = data_scaled.copy()
    data_masked[mask] = np.nan
 
    # Mean imputation (per column)
    imputer = SimpleImputer(strategy="mean")
    data_imputed = imputer.fit_transform(data_masked)
 
    # Squared errors at masked positions only
    sq_err_all = (data_scaled[mask] - data_imputed[mask]) ** 2
    rmse_global = float(np.sqrt(np.mean(sq_err_all))) if sq_err_all.size else np.nan
 
    # Per-group RMSE: restrict mask to a column subset
    def group_rmse(col_idx):
        group_mask = np.zeros_like(mask)
        group_mask[:, col_idx] = mask[:, col_idx]
        if not group_mask.any():
            return np.nan
        sq = (data_scaled[group_mask] - data_imputed[group_mask]) ** 2
        return float(np.sqrt(np.mean(sq)))
 
    return {
        "rmse_global": rmse_global,
        "rmse_beams": group_rmse(beam_idx),
        "rmse_gps": group_rmse(gps_idx),
        "n_masked_cells": int(mask.sum()),
    }
 
 
def run_experiments(clean_data: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Run both scenarios across all proportions and seeds."""
    validate_dataframe(clean_data)
    # Restrict to retained columns and standardize.
    # Fit scaler on the complete data ONCE so all conditions share the same
    # standardization (otherwise the scaler itself becomes a confound).
    data_full = clean_data[RETAINED_COLS].to_numpy(dtype=float)
    scaler = StandardScaler()
    data_scaled = scaler.fit_transform(data_full)
 
    # Column index lookups (positions within RETAINED_COLS / data_scaled)
    col_to_idx = {c: i for i, c in enumerate(RETAINED_COLS)}
    beam_idx = np.array([col_to_idx[c] for c in BEAM_COLS])
    gps_idx = np.array([col_to_idx[c] for c in GPS_COLS])
    all_idx = np.arange(len(RETAINED_COLS))
 
    scenarios = {
        "MCAR-all": all_idx,    # all retained columns eligible
        "MCAR-beams": beam_idx,  # only beam columns eligible
    }
 
    records = []
    for scenario_name, target_idx in scenarios.items():
        for prop in PROPORTIONS:
            for seed in range(N_SEEDS):
                rng = np.random.default_rng(seed)
                mask = ampute_mcar(data_scaled, prop, target_idx, rng)
                metrics = evaluate_one_run(data_scaled, mask, beam_idx, gps_idx)
                records.append({
                    "scenario": scenario_name,
                    "proportion": prop,
                    "seed": seed,
                    **metrics,
                })
 
    results_df = pd.DataFrame(records)
 
    # Aggregate: mean and std across seeds
    summary_df = (
        results_df
        .groupby(["scenario", "proportion"])
        .agg(
            rmse_global_mean=("rmse_global", "mean"),
            rmse_global_std=("rmse_global", "std"),
            rmse_beams_mean=("rmse_beams", "mean"),
            rmse_beams_std=("rmse_beams", "std"),
            rmse_gps_mean=("rmse_gps", "mean"),
            rmse_gps_std=("rmse_gps", "std"),
            avg_masked_cells=("n_masked_cells", "mean"),
            n_seeds=("seed", "count"),
        )
        .round(4)
        .reset_index()
    )
 
    return results_df, summary_df

In [8]:
results_df, summary_df = run_experiments(clean_data)

print(summary_df.to_string(index=False))

  scenario  proportion  rmse_global_mean  rmse_global_std  rmse_beams_mean  rmse_beams_std  rmse_gps_mean  rmse_gps_std  avg_masked_cells  n_seeds
  MCAR-all         0.1            1.0000           0.0112           0.9997          0.0124         1.0023        0.0216          16100.50      100
  MCAR-all         0.3            1.0006           0.0052           1.0005          0.0057         1.0017        0.0101          48278.60      100
  MCAR-all         0.5            1.0000           0.0033           0.9998          0.0037         1.0018        0.0070          80459.80      100
MCAR-beams         0.1            1.0000           0.0127           1.0000          0.0127            NaN           NaN          14717.46      100
MCAR-beams         0.3            1.0006           0.0068           1.0006          0.0068            NaN           NaN          44141.61      100
MCAR-beams         0.5            1.0007           0.0044           1.0007          0.0044            NaN           Na

In [9]:
# Import libraries for experiment driver and external imputers module

import time

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

from imputers.imputers import (
    Imputer,
    MeanImputer,
    KNNImputerWrapper,
    MICEImputer,
    SoftImputeWrapper,
    HyperImputeImputer,
    get_default_imputers,
 )

In [10]:
# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------

PROPORTIONS = [0.10, 0.30, 0.50]
N_SEEDS = 10

BEAM_COLS = [f"beam_{i:02d}" for i in range(64)]
GPS_COLS = [
    "unit2_lat",
    "unit2_lon",
    "unit2_direction",
    "unit2_num_sat",
    "unit2_PDOP",
    "unit2_HDOP",
]
RETAINED_COLS = BEAM_COLS + GPS_COLS


# ---------------------------------------------------------------------------
# Sanity check
# ---------------------------------------------------------------------------

def validate_dataframe(df: pd.DataFrame) -> None:
    missing = [c for c in RETAINED_COLS if c not in df.columns]
    if missing:
        raise KeyError(
            f"clean_data is missing {len(missing)} expected column(s): "
            f"{missing[:5]}{'...' if len(missing) > 5 else ''}"
        )
    if df[RETAINED_COLS].isna().any().any():
        raise ValueError(
            "clean_data already contains NaN values in retained columns. "
            "Amputation experiments require fully observed input."
        )


# ---------------------------------------------------------------------------
# Amputation + evaluation
# ---------------------------------------------------------------------------

def ampute_mcar(shape: tuple[int, int], prop: float,
                target_col_idx: np.ndarray,
                rng: np.random.Generator) -> np.ndarray:
    """Generate an MCAR boolean mask. Only `target_col_idx` columns eligible."""
    mask = np.zeros(shape, dtype=bool)
    sub_mask = rng.random((shape[0], len(target_col_idx))) < prop
    mask[:, target_col_idx] = sub_mask
    return mask


def evaluate_one_run(data_scaled: np.ndarray, mask: np.ndarray,
                     imputer: Imputer, seed: int,
                     beam_idx: np.ndarray, gps_idx: np.ndarray) -> dict:
    """Apply mask -> impute -> compute RMSE on amputed cells."""
    data_masked = data_scaled.copy()
    data_masked[mask] = np.nan

    data_imputed = imputer.fit_transform(data_masked, seed=seed)

    if np.isnan(data_imputed).any():
        col_means = np.nanmean(data_masked, axis=0)
        col_means = np.where(np.isnan(col_means), 0.0, col_means)
        nan_pos = np.isnan(data_imputed)
        data_imputed = data_imputed.copy()
        data_imputed[nan_pos] = np.take(col_means, np.where(nan_pos)[1])

    sq_err_all = (data_scaled[mask] - data_imputed[mask]) ** 2
    rmse_global = float(np.sqrt(np.mean(sq_err_all))) if sq_err_all.size else np.nan

    def group_rmse(col_idx):
        group_mask = np.zeros_like(mask)
        group_mask[:, col_idx] = mask[:, col_idx]
        if not group_mask.any():
            return np.nan
        sq = (data_scaled[group_mask] - data_imputed[group_mask]) ** 2
        return float(np.sqrt(np.mean(sq)))

    return {
        "rmse_global": rmse_global,
        "rmse_beams": group_rmse(beam_idx),
        "rmse_gps": group_rmse(gps_idx),
        "n_masked_cells": int(mask.sum()),
    }


# ---------------------------------------------------------------------------
# Main experiment driver
# ---------------------------------------------------------------------------

def run_experiments(clean_data: pd.DataFrame,
                    imputers: list[Imputer] | None = None,
                    verbose: bool = True) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Run all (scenario x proportion x method x seed) combinations."""
    validate_dataframe(clean_data)

    if imputers is None:
        imputers = get_default_imputers()

    # Standardize once on full clean data (consistent across all conditions).
    data_full = clean_data[RETAINED_COLS].to_numpy(dtype=float)
    scaler = StandardScaler()
    data_scaled = scaler.fit_transform(data_full)

    # Column index lookups
    col_to_idx = {c: i for i, c in enumerate(RETAINED_COLS)}
    beam_idx = np.array([col_to_idx[c] for c in BEAM_COLS])
    gps_idx = np.array([col_to_idx[c] for c in GPS_COLS])
    all_idx = np.arange(len(RETAINED_COLS))

    scenarios = {
        "MCAR-all": all_idx,
        "MCAR-beams": beam_idx,
    }

    total_runs = len(scenarios) * len(PROPORTIONS) * len(imputers) * N_SEEDS
    if verbose:
        print(f"Total runs: {total_runs} "
              f"({len(scenarios)} scenarios x {len(PROPORTIONS)} props "
              f"x {len(imputers)} methods x {N_SEEDS} seeds)")

    records = []
    run_counter = 0
    t_start = time.time()

    for scenario_name, target_idx in scenarios.items():
        for prop in PROPORTIONS:
            for imputer in imputers:
                method_t0 = time.time()
                for seed in range(N_SEEDS):
                    rng = np.random.default_rng(seed)
                    mask = ampute_mcar(data_scaled.shape, prop, target_idx, rng)
                    metrics = evaluate_one_run(
                        data_scaled, mask, imputer, seed, beam_idx, gps_idx
                    )
                    records.append({
                        "scenario": scenario_name,
                        "proportion": prop,
                        "method": imputer.name,
                        "seed": seed,
                        **metrics,
                    })
                    run_counter += 1

                if verbose:
                    elapsed = time.time() - method_t0
                    print(f"  [{run_counter}/{total_runs}] "
                          f"{scenario_name} | p={prop} | {imputer.name}: "
                          f"{elapsed:.1f}s ({elapsed/N_SEEDS:.2f}s/seed)")

    if verbose:
        print(f"\nTotal runtime: {time.time() - t_start:.1f}s")

    results_df = pd.DataFrame(records)

    summary_df = (
        results_df
        .groupby(["scenario", "proportion", "method"])
        .agg(
            rmse_global_mean=("rmse_global", "mean"),
            rmse_global_std=("rmse_global", "std"),
            rmse_beams_mean=("rmse_beams", "mean"),
            rmse_beams_std=("rmse_beams", "std"),
            rmse_gps_mean=("rmse_gps", "mean"),
            rmse_gps_std=("rmse_gps", "std"),
            avg_masked_cells=("n_masked_cells", "mean"),
            n_seeds=("seed", "count"),
        )
        .round(4)
        .reset_index()
    )

    return results_df, summary_df

In [11]:
# Quick smoke-test (single seed, skip MICE for speed)
N_SEEDS = 1
quick_imputers = [
    MeanImputer(),
    KNNImputerWrapper(k=5),
    SoftImputeWrapper(),
 ]
quick_results_df, quick_summary_df = run_experiments(
    clean_data,
    imputers=quick_imputers,
    verbose=False,
 )
print(quick_summary_df[["scenario", "proportion", "method", "rmse_global_mean"]])
print("Any NaN in rmse_global_mean:", quick_summary_df["rmse_global_mean"].isna().any())

      scenario  proportion      method  rmse_global_mean
0     MCAR-all         0.1         KNN            0.2979
1     MCAR-all         0.1        Mean            0.9948
2     MCAR-all         0.1  SoftImpute            0.2641
3     MCAR-all         0.3         KNN            0.3308
4     MCAR-all         0.3        Mean            0.9965
5     MCAR-all         0.3  SoftImpute            0.3094
6     MCAR-all         0.5         KNN            0.4669
7     MCAR-all         0.5        Mean            0.9966
8     MCAR-all         0.5  SoftImpute            0.3982
9   MCAR-beams         0.1         KNN            0.2543
10  MCAR-beams         0.1        Mean            1.0329
11  MCAR-beams         0.1  SoftImpute            0.2065
12  MCAR-beams         0.3         KNN            0.3053
13  MCAR-beams         0.3        Mean            1.0019
14  MCAR-beams         0.3  SoftImpute            0.2522
15  MCAR-beams         0.5         KNN            0.4768
16  MCAR-beams         0.5     

In [12]:
x , y = run_experiments(clean_data=clean_data, imputers=[HyperImputeImputer()], verbose=True)

Total runs: 6 (2 scenarios x 3 props x 1 methods x 1 seeds)


/mnt/c/Users/Kenneth Chan/Documents/School/Research Project/Repo/ImputationFor6GDatasets/imputation_env/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/mnt/c/Users/Kenneth Chan/Documents/School/Research Project/Repo/ImputationFor6GDatasets/imputation_env/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/mnt/c/Users/Kenneth Chan/Documents/School/Research Project/Repo/ImputationFor6GDatasets/imputation_env/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be 

  [1/6] MCAR-all | p=0.1 | HyperImpute: 335.1s (335.15s/seed)


/mnt/c/Users/Kenneth Chan/Documents/School/Research Project/Repo/ImputationFor6GDatasets/imputation_env/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/mnt/c/Users/Kenneth Chan/Documents/School/Research Project/Repo/ImputationFor6GDatasets/imputation_env/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/mnt/c/Users/Kenneth Chan/Documents/School/Research Project/Repo/ImputationFor6GDatasets/imputation_env/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be 

  [2/6] MCAR-all | p=0.3 | HyperImpute: 262.9s (262.85s/seed)


/mnt/c/Users/Kenneth Chan/Documents/School/Research Project/Repo/ImputationFor6GDatasets/imputation_env/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/mnt/c/Users/Kenneth Chan/Documents/School/Research Project/Repo/ImputationFor6GDatasets/imputation_env/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/mnt/c/Users/Kenneth Chan/Documents/School/Research Project/Repo/ImputationFor6GDatasets/imputation_env/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be 

  [3/6] MCAR-all | p=0.5 | HyperImpute: 230.3s (230.32s/seed)
  [4/6] MCAR-beams | p=0.1 | HyperImpute: 308.9s (308.89s/seed)
  [5/6] MCAR-beams | p=0.3 | HyperImpute: 74.9s (74.91s/seed)
  [6/6] MCAR-beams | p=0.5 | HyperImpute: 165.5s (165.55s/seed)

Total runtime: 1377.7s
